**Checking if data is loading from s3 bucket to databricks.**

In [0]:
%fs ls s3a://my-retail-lakehouse/bronze/

**Implementing bronze layer which stores data in a raw form without any transformations. Now with the data extracted from external sources we can transform data in next layer.**

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze_db;


CREATE TABLE IF NOT EXISTS bronze_db.customers
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/customers/",
  header "true"
);
SELECT * FROM bronze_db.customers LIMIT 10;

CREATE TABLE IF NOT EXISTS bronze_db.products
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/products/",
  header "true"
);


CREATE TABLE IF NOT EXISTS bronze_db.stores
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/stores/",
  header "true"
);


CREATE TABLE IF NOT EXISTS bronze_db.sales
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/sales/",
  header "true"
);


/*Verify the data extraction*/
SELECT * FROM bronze_db.products LIMIT 5;
SELECT * FROM bronze_db.stores LIMIT 5;
SELECT * FROM bronze_db.sales LIMIT 5;

**Implementing silver layer for data cleaning and data transformation**

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver_db;


CREATE TABLE silver_db.customers AS
SELECT DISTINCT
    CustomerID,
    INITCAP(TRIM(CustomerName)) AS CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    to_date(LastUpdated, 'dd-MM-yyyy') AS LastUpdated
FROM bronze_db.customers
WHERE CustomerID IS NOT NULL;


**Validating the data tranformation in the silver layer**

In [0]:
%sql
/*Verify Emails*/
SELECT *
FROM silver_db.customers
WHERE Email != LOWER(Email);

/*Verify Extra spaces*/
SELECT *
FROM silver_db.customers
WHERE City LIKE ' %' OR City LIKE '% ';

/*Verify name format*/
SELECT *
FROM silver_db.customers
WHERE CustomerName != INITCAP(CustomerName);


/*Verify date format*/
SELECT *
FROM silver_db.customers
WHERE LastUpdated IS NULL;